In [ ]:
from typing import cast
import pandas as pd
import numpy as np
import bs4
import requests
from typing import Coroutine
import asyncio
import httpx
from typing import Any

In [115]:
url_principal:str='https://www.pisos.com/'
soup:bs4.BeautifulSoup=bs4.BeautifulSoup(requests.get(url=url_principal).text,'html.parser')
links:bs4.ResultSet=soup.find_all(class_='seo-box__location-link--level3')

In [116]:
opciones:list[str]=[opcion+"/" for opcion in ['venta','alquiler']]
tipos:list[str]=[tipo+"-" for tipo in ['pisos']]
tipos

['pisos-']

In [117]:
localidades:list[str]=[link['href'].split('-')[-1] for link in links]
localidades

['roquetas_de_mar/',
 'vera/',
 'almeria_capital/',
 'chiclana_de_la_frontera/',
 'jerez_de_la_frontera/',
 'cadiz_capital/',
 'cordoba_capital_zona_urbana/',
 'area_de_granada_granada_capital/',
 'almunecar/',
 'jaen_capital/',
 'marbella/',
 'estepona/',
 'mijas/',
 'sevilla_capital/',
 'dos_hermanas/',
 'zaragoza_capital/',
 'santander/',
 'salamanca_capital/',
 'valladolid_capital/',
 'albacete_capital_zona_urbana/',
 'barcelona_capital/',
 'sabadell/',
 'badalona/',
 'roses/',
 'castello_empuries/',
 'lloret_de_mar/',
 'calafell/',
 'madrid_capital_zona_urbana/',
 'torrevieja/',
 'orihuela/',
 'pilar_de_la_horadada/',
 'castello_de_la_plana/',
 'valencia_capital_zona_urbana/',
 'gandia/',
 'badajoz_capital/',
 'ourense_capital/',
 'vigo/',
 'palma_de_mallorca/',
 'calvia/',
 'las_palmas_de_gran_canaria/',
 'la_oliva/',
 'adeje/',
 'arona/',
 'logrono/',
 'san_sebastian_donostia/',
 'bilbao/',
 'oviedo/',
 'murcia_capital/',
 'los_alcazares/',
 'san_pedro_del_pinatar/']

In [85]:
len(localidades)

50

In [118]:
urls:list[str]=[url_principal+opcion_tipo_localidad for opcion_tipo_localidad in [opcion+tipo_localidad for tipo_localidad in [tipo+localidad for localidad in localidades for tipo in tipos] for opcion in opciones]]
urls_venta:list[str]=urls[::2]
urls_alquiler:list[str]=urls[1::2]
urls

['https://www.pisos.com/venta/pisos-roquetas_de_mar/',
 'https://www.pisos.com/alquiler/pisos-roquetas_de_mar/',
 'https://www.pisos.com/venta/pisos-vera/',
 'https://www.pisos.com/alquiler/pisos-vera/',
 'https://www.pisos.com/venta/pisos-almeria_capital/',
 'https://www.pisos.com/alquiler/pisos-almeria_capital/',
 'https://www.pisos.com/venta/pisos-chiclana_de_la_frontera/',
 'https://www.pisos.com/alquiler/pisos-chiclana_de_la_frontera/',
 'https://www.pisos.com/venta/pisos-jerez_de_la_frontera/',
 'https://www.pisos.com/alquiler/pisos-jerez_de_la_frontera/',
 'https://www.pisos.com/venta/pisos-cadiz_capital/',
 'https://www.pisos.com/alquiler/pisos-cadiz_capital/',
 'https://www.pisos.com/venta/pisos-cordoba_capital_zona_urbana/',
 'https://www.pisos.com/alquiler/pisos-cordoba_capital_zona_urbana/',
 'https://www.pisos.com/venta/pisos-area_de_granada_granada_capital/',
 'https://www.pisos.com/alquiler/pisos-area_de_granada_granada_capital/',
 'https://www.pisos.com/venta/pisos-almu

In [140]:
async def consultar_n_paginas_opcion_localidad(id:int,u:str,client:httpx.AsyncClient)-> dict[int,int]:
    result:httpx.Response=await client.get(u)
    posible:bs4.Tag | None=bs4.BeautifulSoup(result.text,'html.parser').find(class_='grid__title')
    n:str='0'
    if title:=posible:
        r:str=title.find_all('span')[1].text
        if len(r)>0:
            n=r.split(' ')[0]
    resultados:int=int(n.replace('.',''))
    n_paginas_completas:int=resultados//30
    return {id:int(np.min([n_paginas_completas+(resultados>(30*n_paginas_completas)),100]))}
async def sacar_n_paginas()->dict[int,int]:
    async with httpx.AsyncClient(transport=httpx.AsyncHTTPTransport(retries=1)) as client:
        tareas:list[Coroutine[Any,Any,dict[int,int]]]=[consultar_n_paginas_opcion_localidad(id,u,client) for id,u in enumerate(urls)]
        tareas_completas:list[dict[int,int]]=await asyncio.gather(*tareas)
        return {k:v for d in tareas_completas for k,v in d.items()}
n_paginas:dict[int,int]=await sacar_n_paginas()
n_paginas

{0: 35,
 1: 3,
 2: 32,
 3: 4,
 4: 30,
 5: 13,
 6: 33,
 7: 3,
 8: 32,
 9: 2,
 10: 24,
 11: 4,
 12: 70,
 13: 7,
 14: 100,
 15: 28,
 16: 23,
 17: 8,
 18: 22,
 19: 6,
 20: 100,
 21: 22,
 22: 100,
 23: 8,
 24: 100,
 25: 4,
 26: 65,
 27: 25,
 28: 25,
 29: 2,
 30: 25,
 31: 5,
 32: 19,
 33: 8,
 34: 22,
 35: 27,
 36: 24,
 37: 3,
 38: 31,
 39: 4,
 40: 100,
 41: 35,
 42: 43,
 43: 1,
 44: 35,
 45: 2,
 46: 52,
 47: 1,
 48: 41,
 49: 1,
 50: 31,
 51: 1,
 52: 34,
 53: 1,
 54: 100,
 55: 100,
 56: 100,
 57: 8,
 58: 100,
 59: 6,
 60: 100,
 61: 2,
 62: 35,
 63: 2,
 64: 95,
 65: 45,
 66: 25,
 67: 9,
 68: 30,
 69: 6,
 70: 24,
 71: 6,
 72: 29,
 73: 5,
 74: 68,
 75: 10,
 76: 34,
 77: 6,
 78: 25,
 79: 9,
 80: 21,
 81: 1,
 82: 71,
 83: 5,
 84: 48,
 85: 5,
 86: 20,
 87: 1,
 88: 20,
 89: 6,
 90: 30,
 91: 10,
 92: 22,
 93: 13,
 94: 90,
 95: 14,
 96: 76,
 97: 2,
 98: 70,
 99: 2}

In [159]:
((sum([n_paginas[i] for i in [*n_paginas.keys()][0::2]])*32)/60)/60 # Tarda mas de un día en sacar todos los anuncios de ventas

22.053333333333335

In [142]:
async def consultar_anuncios_cargados(client:httpx.AsyncClient,url:str,pagina:int)->list[str]:
    result:httpx.Response=await client.get(f"{url}{pagina}")
    ads=bs4.BeautifulSoup(result.text,'html.parser').find_all(class_='ad-preview')
    return [url_principal[:-1]+str(ad['data-lnk-href']) for ad in ads]
async def consultar_anuncios_url(client:httpx.AsyncClient,url:str)->list[str]:
    tareas:list[Coroutine[Any,Any,list[str]]]=[consultar_anuncios_cargados(client,url,i) for i in range(1,n_paginas[urls.index(url)]+1)]
    tareas_completas:list[list[str]]=await asyncio.gather(*tareas)
    anuncios:list[str]=[*[u for lista in tareas_completas for u in lista]]
    return anuncios
async def consultar_anuncios(lista_urls:list[str])->list[str]:
    async with httpx.AsyncClient(transport=httpx.AsyncHTTPTransport(retries=1)) as client:
        tareas:list[Coroutine[Any,Any,list[str]]]=[consultar_anuncios_url(client,url) for url in lista_urls]
        tareas_completas:list[list[str]]=await asyncio.gather(*tareas)
        anuncios:list[str]=[*[u for lista in tareas_completas for u in lista]]
    return anuncios

In [139]:
ventas_ads:list[str]=await consultar_anuncios(urls_venta)
ventas_ads

PoolTimeout: 